In [18]:
from torch.nn import RNN

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import einsum
from einops import rearrange

# RNN Maths
Inputs: $\mathbf{x}_t, \ \mathbf{h}_{t-1}$ where $\mathbf{x_t} \in \mathbb{R}^{L\times F}$ and $\mathbf{h}_{t-1} \in \mathbb{R}^{H\times F}$


Parametric Equations (Base):
- $\mathbf{a_t} = \mathbf{b} + \mathbf{Wh_{t-1} + \mathbf{Ux_t}}$
- $\mathbf{h_t} = \tanh{\mathbf{a_t}}$
- $\mathbf{o_t} = \mathbf{c} + \mathbf{Vh_t}$

Parametric Equations (Optimized):
- $\mathbf{a_t} = \mathbf{b} + \mathbf{\hat{W}_t \hat{x}_t}$ where $\mathbf{\hat{W}_t} = \begin{bmatrix}\mathbf{U} \\ \mathbf{W}\end{bmatrix}$ and $\mathbf{\hat{x}_t} = \begin{bmatrix}\mathbf{x_t} \\ \mathbf{h_{t-1}}\end{bmatrix}$
- $\mathbf{h_t} = \tanh{\mathbf{a_t}}$
- $\mathbf{o_t} = \mathbf{c} + \mathbf{Vh_t}$

Loss Functions:
- $\mathcal{L} = -\log{p(y_t | \set{\mathbf{x_1},\mathbf{x_2},...,\mathbf{x_t}})}$
- $\hat{\mathbf{y_t}} = softmax(\mathbf{o_t})$


# Note
> Torch implementation has no hidden-to-output weights (V), apply MLP instead after RNN layer

In [45]:
class EinRnn(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(EinRnn, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

    def init_params(self):
        self.W = [ 
            [ 
                nn.Parameter(torch.randn(self.input_size + self.hidden_size, self.hidden_size) for _ in self.num_layers)
            ] for 
        ]
        # TODO: initialize W_hat from RNN
        #self.U = nn.Parameter(torch.randn(self.hidden_size, self.hidden_size))
        #self.b = nn.Parameter(torch.randn(self.hidden_size))

    def forward(self,x, h0=None):
        # x: (seq_len, batch_size, input_size)
        # h0: (num_layers, batch_size, hidden_size)
        seq_len, batch_size, _ = x.size()
        h = torch.zeros(self.num_layers, batch_size, self.hidden_size) #D,N,H

        for i in self.num_layers:
            for t in range(seq_len):
                x_hat_t = torch.cat([x[t], h[i]], dim=-1)
                h[i] = torch.tanh(torch.matmul(x_hat_t, self.W[i])) 


In [43]:
#W = torch.random(20,10)
x = torch.randn(10, 3, 10) #L,N,F
h = torch.zeros(1,3,20) #D,N,H

#print(x[0].shape,h.shape)
x_hat = torch.cat([x[0], h[0]], dim=-1)
print(x_hat.shape)

torch.Size([3, 30])


In [26]:
for i in range(x.shape[0]):
    #h = torch.tanh(torch.cat((x[i],h),dim=2) @ W)
    print(x[i].shape,h.shape)
    x_hat = torch.cat([rearrange('(1 n) h-> 1 n h',x[i]),h],dim=-2)
    print(x_hat.shape)

torch.Size([3, 10]) torch.Size([1, 3, 20])


RuntimeError: Tensor type unknown to einops <class 'str'>

In [49]:
rnn = RNN(10, 20, 3) # (input_size, hidden_size, num_layers)
x = torch.randn(10, 3, 10) # (seq_len, batch, input_size)

output, hn = rnn(x)
print(output.shape,hn.shape) #seq_len, batch, hidden_size

for _w in rnn.all_weights:
    for w in _w:
        print(w.shape)

torch.Size([10, 3, 20]) torch.Size([3, 3, 20])
torch.Size([20, 10])
torch.Size([20, 20])
torch.Size([20])
torch.Size([20])
torch.Size([20, 20])
torch.Size([20, 20])
torch.Size([20])
torch.Size([20])
torch.Size([20, 20])
torch.Size([20, 20])
torch.Size([20])
torch.Size([20])
